# Titanic | Chargement et audit des donn?es

Ce notebook charge les fichiers de la comp?tition **Titanic - Machine Learning from Disaster**, v?rifie leur structure et ?tablit un diagnostic initial de leur qualit? avant toute pr?paration.

> **P?rim?tre :** les fichiers sources sont lus sans ?tre modifi?s. Le nettoyage, l'analyse approfondie et la mod?lisation sont trait?s dans les notebooks suivants.

**Source :** [Titanic - Machine Learning from Disaster (Kaggle)](https://www.kaggle.com/competitions/titanic)

## 1. P?rim?tre des fichiers

Le jeu d'entra?nement contient la cible `Survived`, connue pour chaque passager. Le jeu de test ne contient pas cette cible : il sert ? produire les pr?dictions. `gender_submission.csv` est un exemple de soumission, et non la v?rit? terrain du jeu de test.

| Fichier | Utilisation | Nombre de lignes attendu |
| --- | --- | ---: |
| `train.csv` | Jeu d'entra?nement avec la cible `Survived`. | 891 |
| `test.csv` | Passagers pour lesquels une pr?diction est attendue. | 418 |
| `gender_submission.csv` | Exemple de pr?dictions associ?es aux identifiants du jeu de test. | 418 |

## 2. Dictionnaire des donn?es | `train.csv`

Chaque ligne repr?sente un passager. `Survived` est la variable cible ? pr?dire.

| Variable | Type attendu | Description |
| --- | --- | --- |
| `PassengerId` | Entier | Identifiant du passager. |
| `Survived` | Binaire | Statut de survie : `0` = non, `1` = oui. |
| `Pclass` | Entier | Classe du billet : 1, 2 ou 3. |
| `Name` | Texte | Nom du passager, titre inclus. |
| `Sex` | Categorie | Sexe indique dans le jeu de donn?es. |
| `Age` | Num?rique | ?ge en annees ; des valeurs peuvent manquer. |
| `SibSp` | Entier | Nombre de fr?res, s?urs ou conjoints ? bord. |
| `Parch` | Entier | Nombre de parents ou d'enfants ? bord. |
| `Ticket` | Texte | R?f?rence du billet. |
| `Fare` | Num?rique | Tarif du billet. |
| `Cabin` | Texte | R?f?rence de cabine, lorsqu'elle est connue. |
| `Embarked` | Cat?gorie | Port d'embarquement : C, Q ou S. |

## 3. Dictionnaire des donn?es | `test.csv`

Le jeu de test reprend les variables descriptives de l'entra?nement, ? l'exception de `Survived`, qui constitue la cible ? pr?dire.

| Variable | Type attendu | Description |
| --- | --- | --- |
| `PassengerId` | Entier | Identifiant du passager. |
| `Pclass` | Entier | Classe du billet : 1, 2 ou 3. |
| `Name` | Texte | Nom du passager, titre inclus. |
| `Sex` | Cat?gorie | Sexe indiqu? dans le jeu de donn?es. |
| `Age` | Num?rique | ?ge en ann?es ; des valeurs peuvent manquer. |
| `SibSp` | Entier | Nombre de fr?res, s?urs ou conjoints ? bord. |
| `Parch` | Entier | Nombre de parents ou d'enfants ? bord. |
| `Ticket` | Texte | R?f?rence du billet. |
| `Fare` | Num?rique | Tarif du billet. |
| `Cabin` | Texte | R?f?rence de cabine, lorsqu'elle est connue. |
| `Embarked` | Cat?gorie | Port d'embarquement : C, Q ou S. |

## 4. Dictionnaire des donn?es | `gender_submission.csv`

Ce fichier fournit une pr?diction de r?f?rence pour chaque passager du jeu de test. Il permet de v?rifier le format d'une soumission, mais pas d'?valuer les pr?dictions sans les ?tiquettes r?elles.

| Variable | Type attendu | Description |
| --- | --- | --- |
| `PassengerId` | Entier | Identifiant du passager du jeu de test. |
| `Survived` | Binaire | Pr?diction de r?f?rence : `0` ou `1`. |

## 5. Environnement et chemins des donn?es

Les fichiers sources se trouvent dans `data/bronze/`. Le dossier du projet est d?tect? ? partir du r?pertoire courant et de ses parents ; l'ex?cution est donc possible depuis la racine du projet ou depuis `notebook/`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (current_dir, *current_dir.parents)
        if (candidate / "data" / "bronze").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Impossible de localiser data/bronze depuis {current_dir}. "
        "Lancez le notebook depuis le projet ou l'un de ses sous-dossiers."
    )

DATA_DIR = PROJECT_ROOT / "data" / "bronze"
SOURCE_FILES = {
    "train": DATA_DIR / "train.csv",
    "test": DATA_DIR / "test.csv",
    "gender_submission": DATA_DIR / "gender_submission.csv",
}

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Dossier des sources : {DATA_DIR}")

## 6. Chargement et validation des fichiers

Chaque fichier est v?rifi? avant lecture. Les contr?les portent sur la pr?sence des colonnes attendues, les identifiants, la cible et la correspondance entre le jeu de test et la baseline.

In [ ]:
def load_dataset(file_path: Path) -> pd.DataFrame:
    """Charge un fichier CSV et explicite les erreurs de lecture."""
    if not file_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {file_path}")

    try:
        frame = pd.read_csv(file_path, low_memory=False)
    except (pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeDecodeError) as exc:
        raise ValueError(f"Impossible de lire le fichier CSV {file_path}: {exc}") from exc

    if frame.empty:
        raise ValueError(f"Le fichier CSV ne contient aucune ligne : {file_path}")

    print(f"{file_path.name} : {frame.shape[0]:,} lignes x {frame.shape[1]} colonnes")
    return frame


def require_columns(frame: pd.DataFrame, required: set[str], dataset_name: str) -> None:
    """Verifie que les colonnes minimales attendues sont presentes."""
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f"{dataset_name} : colonnes requises absentes : {missing}")


df_train = load_dataset(SOURCE_FILES["train"])
df_test = load_dataset(SOURCE_FILES["test"])
df_gender = load_dataset(SOURCE_FILES["gender_submission"])

train_columns = {
    "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp",
    "Parch", "Ticket", "Fare", "Cabin", "Embarked",
}
test_columns = train_columns - {"Survived"}
require_columns(df_train, train_columns, "train.csv")
require_columns(df_test, test_columns, "test.csv")
require_columns(df_gender, {"PassengerId", "Survived"}, "gender_submission.csv")

if "Survived" in df_test.columns:
    raise ValueError("test.csv ne doit pas contenir la cible Survived.")

for frame, dataset_name in (
    (df_train, "train.csv"),
    (df_test, "test.csv"),
    (df_gender, "gender_submission.csv"),
):
    if frame["PassengerId"].isna().any():
        raise ValueError(f"{dataset_name} contient des PassengerId manquants.")
    if frame["PassengerId"].duplicated().any():
        raise ValueError(f"{dataset_name} contient des PassengerId dupliques.")

if set(df_test["PassengerId"]) != set(df_gender["PassengerId"]):
    raise ValueError("Les PassengerId de test.csv et gender_submission.csv ne correspondent pas.")
if not df_train["Survived"].isin([0, 1]).all():
    raise ValueError("Survived doit contenir uniquement 0 ou 1 dans train.csv.")
if not df_gender["Survived"].isin([0, 1]).all():
    raise ValueError("Survived doit contenir uniquement 0 ou 1 dans la baseline.")

print()
print("Structure et identifiants valides ; les fichiers sources restent inchanges.")

## 7. Aper?u des donn?es

Les premi?res lignes permettent de confirmer que les fichiers ont ?t? interpr?t?s comme attendu.

In [ ]:
for dataset_name, frame in (
    ("Entrainement", df_train),
    ("Test", df_test),
    ("Baseline de soumission", df_gender),
):
    print(f"
{dataset_name} - apercu")
    display(frame.head())

## 8. Audit de qualit?

Le rapport par variable pr?sente le type d?tect?, les valeurs pr?sentes et manquantes, le taux de valeurs manquantes et le nombre de valeurs distinctes. Les contr?les signalent ?galement les doublons de lignes, les identifiants dupliqu?s, les ?ges hors de l'intervalle 0-100 ans, les tarifs n?gatifs et les valeurs de cible invalides.

Ces indicateurs sont descriptifs : aucune valeur n'est corrig?e ? ce stade.

In [ ]:
def audit_dataset(
    frame: pd.DataFrame, dataset_name: str
) -> tuple[pd.DataFrame, dict[str, int | None]]:
    """Construit un rapport de qualite sans modifier le jeu de donnees."""
    row_count = len(frame)
    missing_count = frame.isna().sum()
    missing_percentage = (
        (missing_count / row_count * 100).round(2)
        if row_count
        else pd.Series(0.0, index=frame.columns)
    )

    report = pd.DataFrame(
        {
            "Type": frame.dtypes.astype(str),
            "Valeurs presentes": frame.notna().sum(),
            "Valeurs manquantes": missing_count,
            "Manquantes (%)": missing_percentage,
            "Valeurs distinctes": frame.nunique(dropna=True),
        }
    )
    report.index.name = "Variable"

    checks: dict[str, int | None] = {
        "Lignes": row_count,
        "Colonnes": len(frame.columns),
        "Doublons de lignes": int(frame.duplicated().sum()),
        "PassengerId dupliques": (
            int(frame["PassengerId"].duplicated().sum())
            if "PassengerId" in frame.columns
            else None
        ),
    }
    for column, label, lower, upper in (
        ("Age", "Ages hors plage (0-100)", 0, 100),
        ("Fare", "Tarifs negatifs", 0, None),
    ):
        if column not in frame.columns or not pd.api.types.is_numeric_dtype(frame[column]):
            checks[label] = None
        elif upper is None:
            checks[label] = int(frame[column].lt(lower).sum())
        else:
            checks[label] = int((frame[column].lt(lower) | frame[column].gt(upper)).sum())

    checks["Survived hors domaine {0, 1}"] = (
        int((~frame["Survived"].isin([0, 1])).sum())
        if "Survived" in frame.columns
        else None
    )

    print()
    print(f"Audit - {dataset_name} ({row_count:,} lignes, {len(frame.columns)} colonnes)")
    display(report)
    display(pd.DataFrame.from_dict(checks, orient="index", columns=["Resultat"]))
    return report, checks


train_audit, train_checks = audit_dataset(df_train, "train.csv")
test_audit, test_checks = audit_dataset(df_test, "test.csv")
gender_audit, gender_checks = audit_dataset(df_gender, "gender_submission.csv")

In [ ]:
## 9. Synth?se descriptive

Les statistiques descriptives ci-dessous compl?tent l'audit par colonne pour les variables du jeu d'entra?nement. Les valeurs manquantes sont ignor?es dans les statistiques num?riques.

In [ ]:
train_numeric_summary = df_train.describe(include="all").T
display(train_numeric_summary)

## 10. Conclusion et prochaine ?tape

Les tableaux pr?c?dents fournissent un diagnostic reproductible des trois fichiers. Ils permettent d'identifier les colonnes ? examiner en priorit? sans imposer de strat?gie de correction ? ce stade.

**Points de vigilance :** `PassengerId` est un identifiant ? conserver pour le rapprochement des pr?dictions, et non une variable explicative par d?faut. `Survived` est la cible d'entra?nement ; dans la baseline, il s'agit d'une pr?diction et non d'une ?tiquette v?rifi?e.

**?tape suivante :** d?finir et documenter les transformations dans le notebook de pr?paration, en pr?servant les fichiers sources de la couche Bronze.